# 📚 Topics to review — Databricks & SQL study notebook

Built from the handwritten list *"Topics to review"*. Every topic follows the same layout:
**📖 Definition → 🔑 Key points → 💡 Example → 📚 Documentation**.
Cells starting with **▶ Try it** are runnable and self-contained (inline sample data / temp views).

| # | Topic | Runnable demo |
|---|-------|---------------|
| 1 | LEFT JOIN LATERAL (≈ OUTER APPLY) | ✅ SQL |
| 2 | INNER JOIN LATERAL (≈ CROSS APPLY) | ✅ SQL |
| 3 | Recursive CTEs | ✅ SQL (needs DBR 17.0+ / recent Spark) |
| 4 | CDF / CDC | ✅ SQL (creates a scratch Delta table) |
| 5 | Self-joins (use cases) | ✅ SQL |
| 6 | Auto Loader | code snippets |
| 7 | Spark Declarative Pipelines | code snippets |
| 8 | ADF artifacts & self-hosted IR (on-prem) | checklist |
| 9 | Liquid Clustering | code snippets |
| 10 | AQE (Adaptive Query Execution) | ✅ Python + SQL |
| 11 | *(blank in the original list)* | — |

> ⚠️ **Correction to the original list:** items 1 and 2 had the T-SQL equivalents swapped.
> The correct mapping is **`INNER JOIN LATERAL` = `CROSS APPLY`** and **`LEFT JOIN LATERAL` = `OUTER APPLY`**.
> Sections 1–2 use the corrected pairing.

> ℹ️ Doc links point to the Azure Databricks docs on Microsoft Learn (`en-us`); the same pages exist on docs.databricks.com.
> Features marked *Preview* change quickly — always check the "Applies to" banner on the page. Content collected 2026-11-03.

### 🧪 Sample data for the SQL demos (run this first)
Two tiny temp views — nothing is persisted. `employees.manager_id` makes the data hierarchical (Marta is the top manager),
and the **HR** department deliberately has **no employees** so you can see how `INNER` vs `LEFT` behave.

In [0]:
%sql
-- ▶ Try it: create the sample temp views
CREATE OR REPLACE TEMP VIEW departments AS
SELECT * FROM VALUES (1, 'Engineering'), (2, 'Sales'), (3, 'HR')
AS departments(dept_id, dept_name);

CREATE OR REPLACE TEMP VIEW employees AS
SELECT * FROM VALUES
  (100, 'Marta', 1, 15000, CAST(NULL AS INT)),
  (101, 'Piotr', 1, 12000, 100),
  (102, 'Kasia', 1, 11000, 101),
  (103, 'Tomek', 2,  9000, 100),
  (104, 'Ola',   2,  9500, 103),
  (105, 'Ewa',   2,  9500, 103)
AS employees(emp_id, emp_name, dept_id, salary, manager_id);

SELECT * FROM employees ORDER BY emp_id;

---
## 1. LEFT JOIN LATERAL  *(T-SQL: `OUTER APPLY`)*

### 📖 Definition
A **lateral join** lets the subquery on the right side of a join **reference columns of the tables to its left** in the same
`FROM` clause, so the subquery is evaluated *per left row* (a correlated derived table).
With **`LEFT JOIN LATERAL`** every left-hand row is kept: if the subquery returns no rows, the right-hand columns are `NULL`.

### 🔑 Key points
- **Syntax:** `FROM a LEFT JOIN LATERAL (SELECT … FROM b WHERE b.key = a.key …) AS x` — the correlation (`a.key`) lives *inside* the subquery.
  PostgreSQL additionally wants `ON TRUE`; Databricks/Spark don't need it.
- **T-SQL equivalent:** `OUTER APPLY`. Databricks SQL has **no** `APPLY` keyword — always write `LATERAL`.
- **Why not a plain correlated subquery?** A lateral subquery can return **many rows and many columns** and may contain
  `ORDER BY … LIMIT n`, which makes it the standard tool for **top-N per group**, **latest row per key** and per-row table expressions.
- **Join-type family:** `LEFT JOIN LATERAL` keeps unmatched left rows (= `OUTER APPLY`); `[INNER] JOIN LATERAL` drops them (= `CROSS APPLY`, section 2).
- **Chaining:** lateral subqueries can build on earlier aliases: `FROM t1, LATERAL (SELECT c1 + c2 AS a), LATERAL (SELECT a * 2 AS b)`.
- **Restriction:** `NATURAL` can't be combined with `LATERAL` (error `INCOMPATIBLE_JOIN_TYPES`).
- **Performance:** the optimizer tries to *decorrelate* the subquery into a normal join; when it can't, it behaves like a per-row nested loop —
  look at `EXPLAIN` on big tables.
- **Don't confuse with** `LATERAL VIEW explode(...)` — the older Hive-style construct that turns arrays/maps into rows
  (`LATERAL VIEW OUTER` keeps rows whose array is empty).

### 💡 Example — top-2 earners per department (departments without employees still appear)
```sql
SELECT d.dept_name, t.emp_name, t.salary
FROM departments d
LEFT JOIN LATERAL (
  SELECT e.emp_name, e.salary
  FROM employees e
  WHERE e.dept_id = d.dept_id
  ORDER BY e.salary DESC, e.emp_name
  LIMIT 2
) AS t
```
The same idea in T-SQL:
```sql
SELECT d.dept_name, t.emp_name, t.salary
FROM departments d
OUTER APPLY (SELECT TOP (2) e.emp_name, e.salary
             FROM employees e
             WHERE e.dept_id = d.dept_id
             ORDER BY e.salary DESC, e.emp_name) AS t;
```

### 📚 Documentation
- Azure Databricks — JOIN (has `JOIN LATERAL` / `LEFT JOIN LATERAL` examples): https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-qry-select-join
- Apache Spark — LATERAL SUBQUERY: https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-lateral-subquery.html
- T-SQL — FROM clause with `CROSS APPLY` / `OUTER APPLY`: https://learn.microsoft.com/sql/t-sql/queries/from-transact-sql

In [0]:
%sql
-- ▶ Try it: LEFT JOIN LATERAL — note the HR row with NULLs
SELECT d.dept_name, t.emp_name, t.salary
FROM departments d
LEFT JOIN LATERAL (
  SELECT e.emp_name, e.salary
  FROM employees e
  WHERE e.dept_id = d.dept_id
  ORDER BY e.salary DESC, e.emp_name
  LIMIT 2
) AS t
ORDER BY d.dept_name, t.salary DESC, t.emp_name

---
## 2. INNER JOIN LATERAL  *(T-SQL: `CROSS APPLY`)*

### 📖 Definition
Same mechanism as section 1 — a subquery on the right that is evaluated **once per left row** and may reference the left row's columns —
but with **inner-join semantics**: a left row for which the subquery returns **no rows is dropped** from the result.

### 🔑 Key points
- **Syntax:** `FROM a [INNER] JOIN LATERAL (…) AS x` or the comma form `FROM a, LATERAL (…) AS x` (both are inner).
- **T-SQL equivalent:** `CROSS APPLY` (rows without a match disappear, like a cross join with an empty set).

| | `[INNER] JOIN LATERAL` | `LEFT JOIN LATERAL` |
|---|---|---|
| Left row, subquery returns 0 rows | **dropped** | **kept**, right columns `NULL` |
| T-SQL | `CROSS APPLY` | `OUTER APPLY` |
| Pick it when… | you only want parents that have results | you must keep every parent (e.g. report *with* empty groups) |

- **Typical uses:** latest *N* orders per customer, "explode then enrich" with a per-row lookup, calling a table-valued function per row,
  reusing a computed expression without repeating it.
- **Gotcha:** a `WHERE x.col IS NOT NULL` placed after a `LEFT JOIN LATERAL` silently turns it into an inner join.
- **Gotcha:** an *aggregate* lateral subquery (`SELECT count(*) …`) always returns exactly one row, so `INNER` and `LEFT` give the same rows.

### 💡 Example — same query as section 1, inner semantics (HR disappears)
```sql
SELECT d.dept_name, t.emp_name, t.salary
FROM departments d
JOIN LATERAL (
  SELECT e.emp_name, e.salary
  FROM employees e
  WHERE e.dept_id = d.dept_id
  ORDER BY e.salary DESC, e.emp_name
  LIMIT 2
) AS t
```
T-SQL: replace `OUTER APPLY` with `CROSS APPLY`.

### 📚 Documentation
- Azure Databricks — JOIN: https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-qry-select-join
- Apache Spark — LATERAL SUBQUERY: https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-lateral-subquery.html
- T-SQL — FROM clause (APPLY): https://learn.microsoft.com/sql/t-sql/queries/from-transact-sql

In [0]:
%sql
-- ▶ Try it: INNER JOIN LATERAL — compare with section 1 (HR is gone)
SELECT d.dept_name, t.emp_name, t.salary
FROM departments d
JOIN LATERAL (
  SELECT e.emp_name, e.salary
  FROM employees e
  WHERE e.dept_id = d.dept_id
  ORDER BY e.salary DESC, e.emp_name
  LIMIT 2
) AS t
ORDER BY d.dept_name, t.salary DESC, t.emp_name

---
## 3. Recursive CTEs

### 📖 Definition
A **common table expression (CTE)** is a named, temporary result set defined with `WITH` and visible only to the statement that follows.
A **recursive CTE** references *itself*: an **anchor (base) query** produces the first rows, then a **recursive (step) query** runs repeatedly —
each time on the rows produced by the previous iteration — until it returns no new rows. The parts are combined with `UNION ALL`.

### 🔑 Key points
- **Shape:** `WITH RECURSIVE name (cols) AS ( <anchor> UNION ALL <step that joins back to name> ) SELECT … FROM name`
- **Typical uses:** org charts, bills of materials, category trees (variable depth); graph / route traversal; number and date series.
- **Databricks support:** **Databricks Runtime 17.0+** and Databricks SQL. The keyword **`RECURSIVE` is required** (T-SQL just says `WITH`).
  It launched as a Public Preview — check the doc page for the current status.
- **Guard rails:** default `MAX RECURSION LEVEL` is **100** (exceeding it raises `RECURSION_LEVEL_LIMIT_EXCEEDED`);
  the result is capped at **1,000,000 rows** unless a `LIMIT` controls the recursion (`LIMIT` override needs DBR 17.2+; `LIMIT ALL` removes the cap).
- **Limitations:** not supported inside `UPDATE`, `DELETE`, `MERGE`; the step query can't use correlated column references to the CTE.
- **Cycles:** if the data can loop (A→B→A) carry a `path` array and filter with `NOT array_contains(path, next_node)`, or cap the depth.
- **T-SQL comparison:** T-SQL limits depth with `OPTION (MAXRECURSION n)` (default 100, `0` = unlimited);
  Databricks puts the limit in the CTE definition.
- **Alternatives:** fixed, known depth → chained self-joins (section 5); very large graphs → consider GraphFrames or an iterative PySpark loop.

### 💡 Examples
Depth limit (Databricks syntax — fails with `RECURSION_LEVEL_LIMIT_EXCEEDED` because it needs 100 steps):
```sql
WITH RECURSIVE t(n) MAX RECURSION LEVEL 10 AS (
  VALUES (1)
  UNION ALL
  SELECT n + 1 FROM t WHERE n < 100
)
SELECT * FROM t;
```
T-SQL equivalent of the limit: `… SELECT * FROM t OPTION (MAXRECURSION 10);`

### 📚 Documentation
- Azure Databricks — Common table expression (incl. recursive examples & limits): https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-qry-select-cte
- T-SQL — WITH common_table_expression: https://learn.microsoft.com/en-us/sql/t-sql/queries/with-common-table-expression-transact-sql
- T-SQL — Recursive queries using CTEs: https://learn.microsoft.com/en-us/sql/t-sql/queries/recursive-common-table-expression-transact-sql

In [0]:
%sql
-- ▶ Try it: walk the employee hierarchy top-down (anchor = people without a manager)
WITH RECURSIVE hierarchy AS (
  -- anchor
  SELECT emp_id, emp_name, manager_id, 0 AS level, CAST(emp_name AS STRING) AS path
  FROM employees
  WHERE manager_id IS NULL
  UNION ALL
  -- step: attach the direct reports of the previous level
  SELECT e.emp_id, e.emp_name, e.manager_id, h.level + 1, concat(h.path, ' > ', e.emp_name)
  FROM employees e
  JOIN hierarchy h ON e.manager_id = h.emp_id
)
SELECT level, emp_name, path
FROM hierarchy
ORDER BY path

In [0]:
%sql
-- ▶ Try it: cycle-safe graph traversal (the data contains a loop: New York -> Boston -> New York)
CREATE OR REPLACE TEMP VIEW routes(origin, destination) AS VALUES
  ('New York', 'Washington'),
  ('New York', 'Boston'),
  ('Boston', 'New York'),
  ('Washington', 'Boston'),
  ('Washington', 'Raleigh');

WITH RECURSIVE destinations_from_new_york AS (
  SELECT 'New York' AS destination, ARRAY('New York') AS path, 0 AS length
  UNION ALL
  SELECT r.destination, CONCAT(d.path, ARRAY(r.destination)), d.length + 1
  FROM routes AS r
  JOIN destinations_from_new_york AS d
    ON d.destination = r.origin
   AND NOT ARRAY_CONTAINS(d.path, r.destination)   -- cycle guard
)
SELECT * FROM destinations_from_new_york ORDER BY length, destination

---
## 4. CDF / CDC

### 📖 Definition
- **CDC — Change Data Capture:** the general *pattern / technology* of capturing row-level **inserts, updates and deletes** in a source system
  and shipping only those changes downstream instead of reloading full snapshots (SQL Server CDC, Debezium, GoldenGate, …).
- **CDF — Change Data Feed:** the **Delta Lake feature** on Databricks that records row-level changes **between versions of a Delta table**,
  so downstream consumers can read just what changed.

| | CDC | CDF |
|---|---|---|
| What it is | capture pattern / source-side technology | Delta table feature |
| Produces | a change feed (op type + ordering column) | `table_changes()` / `readChangeFeed` output |
| Metadata columns | whatever the tool emits | `_change_type`, `_commit_version`, `_commit_timestamp` |
| Typical place in the flow | source DB → landing / Kafka | silver/gold Delta table → downstream tables |

### 🔑 Key points
- **Enable (legacy CDF):** per table `delta.enableChangeDataFeed = true` (or default for *new* tables:
  `spark.databricks.delta.properties.defaults.enableChangeDataFeed = true`). Only changes made **after** enabling are recorded.
- **`_change_type` values:** `insert`, `update_preimage` (row before), `update_postimage` (row after), `delete`.
- **Read in batch:** `SELECT * FROM table_changes('t', start_version[, end_version])` or
  `spark.read.option("readChangeFeed","true").option("startingVersion", n).table("t")` (start/end inclusive; timestamps also work).
- **Read as a stream:** `spark.readStream.option("readChangeFeed","true").table("t")` — first start returns the current snapshot as `insert`s,
  then future changes; Structured Streaming tracks the versions for you.
- **Not a permanent history:** change records are cleaned up with the table history (`VACUUM`, log retention).
  Need an audit trail? Stream the feed into another table (e.g. with `trigger(availableNow=True)`).
- **Schema changes:** a range that contains a non-additive change (rename/drop column, type change) can't be read — split the query around it.
- **Newer: *automatic* CDF (Public Preview):** computes changes at **read time** from row tracking (Delta) / row lineage (Iceberg v3) —
  no per-table setting, cheaper `MERGE`/`UPDATE` writes. Needs Unity Catalog and a recent Databricks Runtime; you can't mix it with legacy CDF on one table.
- **CDF vs `AUTO CDC`:** `AUTO CDC` (formerly `APPLY CHANGES INTO`) is a **Spark Declarative Pipelines** API that *consumes* a change feed
  (from CDF, Debezium, …) and applies it to a target as **SCD type 1 or 2**, handling out-of-order events via `SEQUENCE BY`.
  Databricks recommends it over hand-written CDF + `MERGE` for SCD1/2 (section 7). It's Databricks-only (not in open-source Spark SDP).

### 💡 Example
```sql
-- enable on an existing table
ALTER TABLE main.sales.orders SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- batch: all changes from version 5 to 10
SELECT * FROM table_changes('main.sales.orders', 5, 10);
```
```python
# streaming read of the change feed
changes = (spark.readStream
    .option("readChangeFeed", "true")
    .table("main.sales.orders"))
```

### 📚 Documentation
- Use change data feed on Azure Databricks: https://learn.microsoft.com/en-us/azure/databricks/tables/features/change-data-feed
- The AUTO CDC APIs (SCD type 1 / 2 in pipelines): https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc
- AUTO CDC INTO (SQL reference): https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/ldp-sql-ref-apply-changes-into

In [0]:
%sql
-- ▶ Try it: creates a scratch Delta table in your CURRENT catalog/schema (needs CREATE TABLE there)
DROP TABLE IF EXISTS cdf_demo;
CREATE TABLE cdf_demo (id INT, name STRING) TBLPROPERTIES (delta.enableChangeDataFeed = true);  -- version 0
INSERT INTO cdf_demo VALUES (1, 'alice'), (2, 'bob');   -- version 1
UPDATE cdf_demo SET name = 'BOB' WHERE id = 2;          -- version 2
DELETE FROM cdf_demo WHERE id = 1;                      -- version 3

-- Expect: insert x2 (v1), update_preimage + update_postimage (v2), delete (v3)
SELECT id, name, _change_type, _commit_version
FROM table_changes('cdf_demo', 1)
ORDER BY _commit_version, id, _change_type
-- Clean up afterwards:  DROP TABLE IF EXISTS cdf_demo;

---
## 5. Self-joins (use cases)

### 📖 Definition
A **self-join** joins a table **to itself**. You give the table two aliases and treat them as two logical copies,
so a row can be compared with, or linked to, *other rows of the same table*.

### 🔑 Key points — use cases
| Use case | Pattern |
|---|---|
| **Hierarchies** (employee → manager, category → parent) | `e LEFT JOIN t m ON e.manager_id = m.emp_id` — one level per join; unknown depth → recursive CTE (section 3) |
| **Row-vs-row comparison** ("earns more than their manager") | self-join + `WHERE e.salary > m.salary` |
| **Pairs / duplicate detection** | join on the matching columns and use `a.id < b.id` to drop self-pairs and mirrored pairs |
| **Adjacent / previous row** (gaps, deltas) | possible, but **prefer `LAG`/`LEAD` window functions** — usually cheaper and clearer |
| **Graph edges** (friends-of-friends, connections) | one hop per self-join of the edge table |
| **Time / range proximity** (events within *n* minutes) | self-join with a range condition — watch for row explosion and skew |
| **Compare a table with its own past** | join the table to a time-travel version of itself (`VERSION AS OF`) to diff changes |

**Pitfalls**
- Always alias (`e`, `m`) and qualify every column — otherwise columns are ambiguous.
- Pick the join type on purpose: `LEFT` keeps roots (e.g. the CEO with no manager), `INNER` drops them.
- A missing/weak `ON` condition silently becomes a cross join (n² rows).
- `NULL` or very frequent join-key values cause skew (see AQE, section 10).
- Use `a.id < b.id` (not `<>`) for unordered pairs, or you get every pair twice.

### 💡 Examples
```sql
-- employee + manager name
SELECT e.emp_name AS employee, m.emp_name AS manager
FROM employees e LEFT JOIN employees m ON e.manager_id = m.emp_id;

-- employees earning more than their manager
SELECT e.emp_name, e.salary, m.emp_name AS manager, m.salary AS manager_salary
FROM employees e JOIN employees m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary;

-- pairs in the same department with the same salary (each pair once)
SELECT a.emp_name AS emp_a, b.emp_name AS emp_b, a.salary
FROM employees a JOIN employees b
  ON a.dept_id = b.dept_id AND a.salary = b.salary AND a.emp_id < b.emp_id;
```

### 📚 Documentation
- Azure Databricks — JOIN: https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-qry-select-join
- Azure Databricks — Window functions (`LAG` / `LEAD` alternative): https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-window-functions

In [0]:
%sql
-- ▶ Try it: employee -> manager (LEFT JOIN keeps Marta, who has no manager)
SELECT e.emp_name AS employee, m.emp_name AS manager
FROM employees e
LEFT JOIN employees m ON e.manager_id = m.emp_id
ORDER BY e.emp_id

In [0]:
%sql
-- ▶ Try it: who earns more than their own manager?
SELECT e.emp_name AS employee, e.salary, m.emp_name AS manager, m.salary AS manager_salary
FROM employees e
JOIN employees m ON e.manager_id = m.emp_id
WHERE e.salary > m.salary
ORDER BY e.emp_name

In [0]:
%sql
-- ▶ Try it: same-department, same-salary pairs — a.emp_id < b.emp_id yields each pair once
SELECT a.emp_name AS emp_a, b.emp_name AS emp_b, a.salary
FROM employees a
JOIN employees b
  ON a.dept_id = b.dept_id
 AND a.salary  = b.salary
 AND a.emp_id  < b.emp_id

---
## 6. Auto Loader

### 📖 Definition
**Auto Loader** is Databricks' incremental file-ingestion source for cloud storage. It is a Structured Streaming source named **`cloudFiles`**
that discovers **new files as they arrive** (ADLS Gen2, S3, GCS, Unity Catalog volumes), processes each file **exactly once**,
and can **infer and evolve the schema** — without re-listing or re-reading what was already ingested.

### 🔑 Key points
- **File detection modes** (switchable across restarts, still exactly-once):
  - *Directory listing* — default; lists the input path; needs no extra permissions.
  - *File notification* — uses cloud events + a queue; more scalable. **Recommended: file notification with *file events*** on a Unity Catalog external location
    (`cloudFiles.useManagedFileEvents`).
- **Formats** (`cloudFiles.format`): `json`, `csv`, `xml`, `parquet`, `avro`, `orc`, `text`, `binaryFile`.
- **State lives in the checkpoint** — one checkpoint per ingestion stream; don't share it between sources.
- **Schema inference & evolution:** set **`cloudFiles.schemaLocation`** (can equal the checkpoint path).
  JSON/CSV/XML columns are inferred as **strings** unless **`cloudFiles.inferColumnTypes = true`**; override with `cloudFiles.schemaHints`.
- **Rescued data column:** unparsable or unexpected data lands in **`_rescued_data`** instead of failing the stream.
- **Schema evolution modes** (`cloudFiles.schemaEvolutionMode`): `addNewColumns` (default when no schema is given — the stream **stops with `UnknownFieldException`**
  and picks up the new schema on restart), `rescue`, `failOnNewColumns`, `none`; plus `addNewColumnsWithTypeWidening` (Preview).
  → run it under a Lakeflow Job with retries so restarts are automatic.
- **Trigger choice:** continuous micro-batches by default, or **`trigger(availableNow=True)`** for scheduled *incremental batch* runs.
- **No ordering guarantee:** files may be discovered/processed out of order — design downstream logic for it.
- **SQL flavour:** `STREAM read_files(...)` inside `CREATE STREAMING TABLE` uses Auto Loader under the hood.
- **Inside Spark Declarative Pipelines** (section 7) Databricks manages schema location and checkpoints for you.
- **vs `COPY INTO`:** `COPY INTO` is a simple idempotent SQL batch loader; Auto Loader scales better for large / continuous / evolving sources.

### 💡 Example (template — replace the `<…>` paths)
```python
(spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")                    # source file format
    .option("cloudFiles.schemaLocation", "<schema-path>")   # turns on schema inference + evolution
    .option("cloudFiles.inferColumnTypes", "true")          # otherwise JSON/CSV columns are strings
    .load("<landing-path>")                                 # abfss://…  or  /Volumes/<catalog>/<schema>/<volume>/…
  .writeStream
    .option("checkpointLocation", "<checkpoint-path>")
    .trigger(availableNow=True)                             # incremental batch; drop for continuous streaming
    .toTable("<catalog>.<schema>.<bronze_table>"))
```
```sql
-- SQL equivalent as a streaming table
CREATE OR REFRESH STREAMING TABLE bronze_orders
AS SELECT * FROM STREAM read_files('/Volumes/<catalog>/<schema>/<volume>/orders/', format => 'json');
```

### 📚 Documentation
- Auto Loader overview: https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/
- File detection modes (listing vs notification): https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/file-detection-modes
- Schema inference & evolution: https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/schema
- All `cloudFiles.*` options: https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/options
- Directory listing mode: https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/directory-listing-mode
- File notification mode: https://learn.microsoft.com/en-us/azure/databricks/ingestion/cloud-object-storage/auto-loader/file-notification-mode

---
## 7. Spark Declarative Pipelines (SDP)

### 📖 Definition
**Spark Declarative Pipelines** is a declarative framework for building **batch and streaming data pipelines in SQL and Python**.
You declare **what** datasets should exist and how each is computed; the engine works out **dependency order, orchestration, incremental processing,
checkpoints and retries**. On Databricks it is delivered as **Lakeflow Spark Declarative Pipelines** (formerly *Delta Live Tables*), built on the
open-source **Apache Spark Declarative Pipelines** (available from Spark 4.1 as `pyspark.pipelines`).

### 🔑 Key points
- **Building blocks**
  - **Streaming table** — incremental processing of an append-style source (ingestion, Auto Loader, Kafka). Python: `@dp.table` returning a *streaming* DataFrame.
  - **Materialized view** — precomputed result of a *batch* query, refreshed (incrementally when possible). Python: `@dp.materialized_view`.
  - **Temporary view** — intermediate step, not published outside the pipeline (`@dp.temporary_view`). *Private* tables are also available.
  - **Flow** — the query that reads a source and writes a target (append flows, AUTO CDC flows, …). **Sinks** write to Delta, Kafka, Event Hubs.
- **Declarative, not procedural:** decorators / `CREATE` statements only *register* datasets; the engine builds the dataflow graph (DAG) and runs it.
  Functions must return a DataFrame and avoid side effects.
- **Pipeline** = source files + settings (target catalog/schema, compute, configuration). Reference other datasets simply by name.
- **Added by Lakeflow (not in open-source SDP):** `AUTO CDC` (SCD type 1 / 2, + `AUTO CDC FROM SNAPSHOT`), **data-quality expectations**
  (`@dp.expect`, `@dp.expect_or_drop`, `@dp.expect_or_fail`), queryable **event log**, continuous mode, `foreachBatch` sinks.
- **Import convention:** `from pyspark import pipelines as dp` (older code used `import dlt`).
- **Plan:** Lakeflow SDP requires the **Premium** plan on Azure Databricks.
- **Fits with the other topics:** Auto Loader ingests files into streaming tables; `AUTO CDC` implements CDC (section 4); Liquid Clustering (section 9)
  is recommended for new pipeline tables.

### 💡 Example (Python) — bronze streaming table with a quality rule + a gold materialized view
```python
from pyspark import pipelines as dp

@dp.table()
@dp.expect_or_drop("valid_order", "order_id IS NOT NULL")     # drop rows that violate the rule
def orders_bronze():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "json")
            .load("<landing-path>"))

@dp.materialized_view()
def orders_per_day():
    return (spark.read.table("orders_bronze")
            .groupBy("order_date").count())
```
```sql
-- SQL equivalent
CREATE OR REFRESH STREAMING TABLE orders_bronze (
  CONSTRAINT valid_order EXPECT (order_id IS NOT NULL) ON VIOLATION DROP ROW
)
AS SELECT * FROM STREAM read_files('<landing-path>', format => 'json');

CREATE MATERIALIZED VIEW orders_per_day AS
SELECT order_date, count(*) AS orders FROM orders_bronze GROUP BY order_date;
```

### 📚 Documentation
- Lakeflow Spark Declarative Pipelines (landing page): https://learn.microsoft.com/en-us/azure/databricks/ldp
- Concepts — Spark Declarative Pipelines & what Lakeflow adds: https://learn.microsoft.com/en-us/azure/databricks/ldp/concepts/spark-declarative-pipelines
- Develop pipeline code with Python: https://learn.microsoft.com/en-us/azure/databricks/ldp/developer/python-dev
- AUTO CDC APIs: https://learn.microsoft.com/en-us/azure/databricks/ldp/cdc
- Apache Spark — Declarative Pipelines Programming Guide: https://spark.apache.org/docs/latest/declarative-pipelines-programming-guide.html

---
## 8. ADF artifacts (i.e. configuration of the on-prem Integration Runtime)

### 📖 Definition
**Azure Data Factory (ADF)** is Azure's managed data-integration / orchestration service. Its building blocks ("artifacts") are stored as JSON definitions.
The **Integration Runtime (IR)** is the compute and network bridge ADF uses to run activities; a **self-hosted IR (SHIR)** is an agent installed on a machine
**inside your private / on-premises network** so ADF can reach data stores that aren't publicly accessible.

### 🔑 Key points — the main ADF artifacts
| Artifact | What it is |
|---|---|
| **Pipeline** | logical group of activities that together perform a task |
| **Activity** | a processing step: *data movement* (Copy), *transformation* (Data Flow, Databricks Notebook/Jar/Python, …), *control* (ForEach, If, Web, …) |
| **Dataset** | named view of data (location + structure) used as an activity's input/output |
| **Linked service** | connection definition (≈ connection string) to a data store or compute; its **`connectVia`** property names the IR to use |
| **Integration runtime** | compute + connectivity; types: **Azure** (default, auto-resolve), **Self-hosted**, **Azure-SSIS** |
| **Trigger** | decides *when* a pipeline runs (schedule, tumbling window, event, manual) |
| **Mapping data flow** | visually designed, Spark-based transformation |

With Git integration each artifact is saved as a JSON file in its own folder (e.g. `pipeline/`, `dataset/`, `linkedService/`, `integrationRuntime/`, `trigger/`).

### 🔑 Configuring a self-hosted IR for on-prem sources — checklist
1. **Create the IR in ADF:** *Manage → Integration runtimes → + New → Azure, Self-Hosted → Self-Hosted*; copy one of the two **authentication keys**
   (PowerShell: `Get-AzDataFactoryV2IntegrationRuntimeKey`).
2. **Prepare the host:** 64-bit Windows 10/11 or Server 2016/2019/2022/2025 (**not** a domain controller), .NET Framework 4.7.2+, minimum 4 cores / 8 GB RAM / 80 GB disk,
   local admin rights, **hibernation off**. Parquet/ORC/Avro need **JRE/JDK 11** (`JAVA_HOME`). Prefer a **different machine than the data source**; one SHIR instance per machine.
3. **Install & register:** run the MSI, paste the key in *Integration Runtime Configuration Manager → Register* (headless: `dmgcmd -rn "<key>" "<NodeName>"`).
4. **Network — outbound only, no inbound ports from the internet:** HTTPS **443** to `*.servicebus.windows.net` (interactive authoring via Azure Relay),
   `<factory>.<region>.datafactory.azure.net` (or `*.frontend.clouddatahub.net`), `download.microsoft.com` (auto-update), Key Vault; plus source/sink-specific
   endpoints (e.g. `*.core.windows.net` 443, `*.database.windows.net` 1433). Behind a proxy configure it under *Settings → HTTP Proxy*.
5. **Credentials:** prefer **Azure Key Vault**; otherwise they are stored locally, encrypted with Windows DPAPI. Back up the node with `dmgcmd -gbf`.
6. **High availability / scale:** register **up to 4 nodes** to the same IR (enable *Remote access to intranet*, default port **8060**, on the first node); scale up via max concurrent jobs.
7. **Create the linked service** for the on-prem source (SQL Server, file share, Oracle, …), set **`connectVia` = your SHIR**, and **Test connection**.
8. **Share it** with other factories in the same Microsoft Entra tenant via a *linked IR* (not possible across Synapse workspaces).
9. **Monitor:** Configuration Manager status (*Connected* / *Connecting* / *Disconnected* / *Limited* = Azure Relay problem), Windows Event Viewer →
   *Microsoft Integration Runtime*, and ADF *Monitor → Integration runtimes*. "Failed to register this node" usually means firewall/proxy.

### 💡 Typical pattern with Databricks
On-prem SQL Server **→ (ADF Copy activity via SHIR) →** ADLS Gen2 landing zone **→ (Auto Loader / Declarative Pipelines) →** bronze Delta table.
ADF can also trigger Databricks notebooks/jobs through a **Databricks linked service**.

> **Fabric note:** Fabric Data Factory has **no integration runtimes, linked services or datasets** (connections are defined inline). For on-prem access you use the
> **on-premises data gateway (OPDG)** instead of a SHIR. Azure-SSIS IR has no Fabric equivalent yet.

### 📚 Documentation
- Create and configure a self-hosted integration runtime: https://learn.microsoft.com/en-us/azure/data-factory/create-self-hosted-integration-runtime
- Integration runtime concepts: https://learn.microsoft.com/en-us/azure/data-factory/concepts-integration-runtime
- Pipelines and activities: https://learn.microsoft.com/en-us/azure/data-factory/concepts-pipelines-activities
- Linked services: https://learn.microsoft.com/en-us/azure/data-factory/concepts-linked-services
- Fabric: ADF → Fabric feature comparison (SHIR vs OPDG): https://learn.microsoft.com/en-us/fabric/data-factory/activity-parity
- Fabric: access on-premises data with the gateway: https://learn.microsoft.com/en-us/fabric/data-factory/how-to-access-on-premises-data

---
## 9. Liquid Clustering

### 📖 Definition
**Liquid clustering** is Delta Lake's flexible data-layout technique that **replaces Hive-style partitioning and `ZORDER`**.
You declare **clustering keys** with `CLUSTER BY`; Databricks co-locates rows with similar key values to maximise **data skipping**,
and you can **change the keys later without rewriting the whole table**.

### 🔑 Key points
- **Recommended for all new tables**, including streaming tables and materialized views. Best for: filters on high-cardinality columns, skewed data,
  fast-growing tables, concurrent writes, and access patterns that change over time.
- **Not compatible** with partitioning or `ZORDER` on the same table; Databricks manages layout and optimization.
- **Up to 4 clustering keys**; keys need collected statistics (first 32 columns by default). Struct / map / array columns can't be keys (struct *fields* can).
- **Automatic liquid clustering — `CLUSTER BY AUTO`:** Databricks picks and adapts keys from the query history (Unity Catalog **managed** tables,
  predictive optimization, DBR 15.4 LTS+). Docs recommend it with predictive optimization for all managed tables.
- **How data gets clustered:** on write when the transaction is large enough (e.g. ≥ 64 MB for 1 key on managed tables), via **incremental `OPTIMIZE`**,
  automatically by **predictive optimization**, or a full re-cluster with **`OPTIMIZE … FULL`** (needed after enabling clustering or changing keys on existing data).
- **Changing keys:** `ALTER TABLE … CLUSTER BY (new_cols)` applies to new writes/OPTIMIZE immediately; old files keep their layout until `OPTIMIZE FULL`.
  `CLUSTER BY NONE` turns clustering off.
- **Convert a partitioned table:** `ALTER TABLE … REPLACE PARTITIONED BY WITH CLUSTER BY (…)` (DBR 18.1+).
- **Hierarchical clustering** (DBR 17.1+): `delta.liquid.hierarchicalClusteringColumns` prioritises low-cardinality keys.
- **Requirements & caveats:** Delta GA on DBR 15.4 LTS+; Iceberg needs DBR 16.4 LTS+ (Preview). Clustered tables use Delta writer v7 / reader v3 — older clients can't read them
  and protocol versions can't be downgraded. With DataFrame writes, keys can only be set at creation or in `overwrite` mode — otherwise use `ALTER TABLE`.

| | Partitioning + ZORDER | Liquid clustering |
|---|---|---|
| Define | `PARTITIONED BY` + `OPTIMIZE ZORDER BY` | `CLUSTER BY (…)` / `CLUSTER BY AUTO` |
| Change keys later | rewrite the table | `ALTER TABLE … CLUSTER BY` (+ `OPTIMIZE FULL` for old data) |
| High-cardinality keys | small-files / too many partitions | fine |
| Concurrent writes | more conflicts | row-level concurrency, fewer conflicts |

### 💡 Example
```sql
-- new table with explicit keys
CREATE TABLE sales (order_id BIGINT, customer_id BIGINT, order_date DATE, amount DECIMAL(18,2))
CLUSTER BY (order_date, customer_id);

-- let Databricks choose the keys (Unity Catalog managed table)
CREATE OR REPLACE TABLE sales_auto (order_id BIGINT, order_date DATE) CLUSTER BY AUTO;

-- enable / change on an existing (unpartitioned) Delta table
ALTER TABLE sales CLUSTER BY (customer_id, order_date);

-- cluster data: incremental, or full re-cluster after changing keys
OPTIMIZE sales;
OPTIMIZE sales FULL;

-- turn off
ALTER TABLE sales CLUSTER BY NONE;
```

### 📚 Documentation
- Use liquid clustering for tables: https://learn.microsoft.com/en-us/azure/databricks/delta/clustering
- CLUSTER BY clause (SQL reference): https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-syntax-ddl-cluster-by

---
## 10. AQE — Adaptive Query Execution

### 📖 Definition
**Adaptive Query Execution** is Spark SQL **re-optimizing the query plan *while it runs***, using **runtime statistics** collected at the end of each
shuffle/broadcast stage ("query stage"). That makes it robust when table statistics are missing/stale or when data skew shows up mid-query.

### 🔑 Key points
- **On by default** (Spark ≥ 3.2 and Databricks) — no query changes needed.
- **Four main features**
  1. **Sort-merge join → broadcast hash join** at runtime when one side turns out small.
  2. **Coalesce shuffle partitions** — merges many tiny post-shuffle partitions into reasonably sized ones.
  3. **Skew-join handling** — splits (and replicates) oversized partitions in sort-merge / shuffle-hash joins.
  4. **Empty-relation propagation** — prunes parts of the plan that return no rows.
- **Applies to** queries that are **not streaming** and contain at least one **exchange** (join, aggregate, window) and/or a subquery.
- **How to see it:** the plan shows `AdaptiveSparkPlan` with `isFinalPlan=false` before/while running and `true` once finished (Spark UI → SQL/DataFrame tab).
  Plain `EXPLAIN` doesn't execute the query, so it only shows the *initial* plan.
- **Key settings** (defaults as documented for Databricks):

| Setting | Default | Meaning |
|---|---|---|
| `spark.sql.adaptive.enabled` (OSS) / `spark.databricks.optimizer.adaptive.enabled` (Databricks) | `true` | AQE on/off |
| `spark.sql.adaptive.coalescePartitions.enabled` | `true` | coalesce small shuffle partitions |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | `64MB` | target size after coalescing |
| `spark.sql.adaptive.skewJoin.enabled` | `true` | skew-join handling |
| `spark.sql.adaptive.skewJoin.skewedPartitionFactor` | `5` | skewed if > factor × median partition size … |
| `spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes` | `256MB` | … **and** larger than this |
| `spark.databricks.adaptive.autoBroadcastJoinThreshold` | `30MB` | runtime switch to broadcast join |
| `spark.sql.shuffle.partitions` | `200` (`auto` = auto-optimized shuffle) | initial shuffle partitions |

- **FAQ highlights**
  - *Still use a broadcast hint?* **Yes**, if you know the data — a statically planned broadcast usually beats AQE's runtime switch (AQE may only switch after shuffling both sides).
  - *Skew hint or AQE skew handling?* Prefer **AQE** (automatic, usually faster).
  - *Why no skew detection?* A partition must exceed **both** the 256 MB threshold **and** 5 × the median; and only **shuffle-based** joins are handled
    (never broadcast). Which side is optimized depends on the join type (e.g. `LEFT OUTER` → left side only; `FULL OUTER` → none).
  - *Does AQE reorder joins?* **No** — collect table statistics so the cost-based optimizer can order joins.
  - *Why wasn't a small table broadcast?* The join type may not allow broadcasting that side (e.g. the left side of a `LEFT OUTER JOIN`),
    or too many empty partitions.

### 📚 Documentation
- Adaptive query execution on Azure Databricks: https://learn.microsoft.com/en-us/azure/databricks/optimizations/aqe
- Apache Spark — Performance tuning → Adaptive Query Execution: https://spark.apache.org/docs/latest/sql-performance-tuning.html#adaptive-query-execution

In [0]:
# ▶ Try it: inspect the AQE settings of your cluster (some settings can't be read on serverless compute -> shown as n/a)
for key in [
    "spark.sql.adaptive.enabled",
    "spark.databricks.optimizer.adaptive.enabled",
    "spark.sql.adaptive.coalescePartitions.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.shuffle.partitions",
]:
    try:
        print(f"{key:<55} = {spark.conf.get(key)}")
    except Exception:
        print(f"{key:<55} = n/a")

In [0]:
%sql
-- ▶ Try it: look for 'AdaptiveSparkPlan ... isFinalPlan=false' in the output (EXPLAIN doesn't execute the query)
EXPLAIN FORMATTED
SELECT k, count(*) AS cnt
FROM (SELECT id % 10 AS k FROM range(1000000))
GROUP BY k

---
## 11. *(blank in the original list)*

Add your own topic here: **📖 Definition → 🔑 Key points → 💡 Example → 📚 Documentation**.